# 07 — Business Report

Цель ноутбука: собрать понятный отчёт для тимлида: проблема, решение, пример рекомендации, качество модели, загрузка команды и fairness.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

## Problem

Тимлиду сложно вручную выбирать исполнителя задачи: нужно учитывать навыки, загрузку, срочность, качество прошлых работ и развитие сотрудников.

## Solution

COMPASS AI анализирует задачу и команду, ранжирует кандидатов через TaskEmployeeMatchingNet и объясняет рекомендацию на русском языке.

In [ ]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
tasks = pd.read_csv(DATA_SYNTHETIC / "tasks.csv")
assignments = pd.read_csv(DATA_SYNTHETIC / "assignments.csv")

summary = {
    "employees": len(employees),
    "tasks": len(tasks),
    "assignments": len(assignments),
    "success_rate": float(assignments["success_label"].mean()),
    "average_workload": float(employees["current_workload"].mean()),
}

display(summary)

In [ ]:
with open(REPORTS / "model_metrics.json", "r", encoding="utf-8") as file:
    model_metrics = json.load(file)

with open(REPORTS / "ranking_metrics.json", "r", encoding="utf-8") as file:
    ranking_metrics = json.load(file)

display(pd.DataFrame([model_metrics]))
display(pd.DataFrame(ranking_metrics).T)

In [ ]:
from src.agents.orchestrator import recommend_synthetic_task

example = recommend_synthetic_task(
    "TASK-0001",
    mode="balanced_workload",
    top_k=3,
    use_llm=False,
)

print("Task:", example["title"])
print("Mode:", example["mode"])
print("Recommended:", example["top_candidates"][0]["name"])
print("Explanation:")
print(example["explanation"])

In [ ]:
workload_df = (
    employees[["name", "role", "grade", "current_workload", "active_tasks_count"]]
    .sort_values("current_workload", ascending=False)
    .head(15)
)

fig = px.bar(
    workload_df,
    x="name",
    y="current_workload",
    color="role",
    title="Team workload",
)
fig.show()
fig.write_html(FIGURES / "business_team_workload.html")

In [ ]:
fairness_path = REPORTS / "fairness_summary.json"

if fairness_path.exists():
    with open(fairness_path, "r", encoding="utf-8") as file:
        fairness_summary = json.load(file)

    display(fairness_summary)
else:
    print("Run 05_fairness_analysis.ipynb first to create fairness_summary.json")

## Conclusion

COMPASS AI показывает не только accuracy, но и управленческую зрелость:
учитывает риски, загрузку, развитие сотрудников и качество прошлых назначений.